## Step 2: Get Landsat Images from 2000-2020 for Classification

In [1]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config
from helpers import landsat_composites 
import pandas as pd
import ast

In [10]:
noncrop_val = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_nonCrop_manual_validation_samples.csv")
noncrop_val_remove = noncrop_val[noncrop_val["actual"] == 0]["point_id"].tolist()

crop_val = pd.read_csv(r"outputs\phenology_verified_for_validation\final_stable_ag_manual_validation_samples_glad.csv")
crop_val_remove = crop_val[crop_val["actual"] == 0]["point_id"].tolist()

In [13]:
stable_categories_nonCrop = [
    'STABLE NON-CROP (Rangeland)', 
    'STABLE NON-CROP (Forest)', 
    "STABLE NON-CROP (Barren/Water)"
]
noncrop_samples = pd.read_csv(r"outputs\phenology_verified_samples\stable_nonCrop_phenology_classification_re_combined_filtered_2000m.csv")
noncrop_samples = noncrop_samples[noncrop_samples["status"].isin(stable_categories_nonCrop)]
noncrop_samples["stable_crop"] = 0
print(noncrop_samples.shape)
noncrop_samples = noncrop_samples[~noncrop_samples['point_id'].isin(noncrop_val_remove)]
print(noncrop_samples.shape)
noncrop_samples["coords"] = noncrop_samples["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
noncrop_samples["x"] = noncrop_samples["coords"].apply(lambda x: x[0])
noncrop_samples["y"] = noncrop_samples["coords"].apply(lambda x: x[1])
noncrop_samples = noncrop_samples[["stable_crop", "x", "y", "lc2022"]]
forest_df = noncrop_samples[noncrop_samples['lc2022'] == 4]
non_forest_df = noncrop_samples[noncrop_samples['lc2022'] != 4]
sampled_forest_df = forest_df.sample(n=400, random_state=42)
noncrop_balanced_samples = pd.concat([sampled_forest_df, non_forest_df])
print(noncrop_balanced_samples.shape)

crop_samples = pd.read_csv(r"outputs\phenology_verified_samples\stable_ag_phenology_classification_results_glad.csv")
crop_samples = crop_samples[crop_samples["status"] == 'STABLE CROPLAND']
crop_samples["stable_crop"] = 1
print(crop_samples.shape)
crop_samples = crop_samples[~crop_samples['point_id'].isin(crop_val_remove)]
print(crop_samples.shape)
crop_samples["coords"] = crop_samples["geo"].apply(lambda x: ast.literal_eval(x)["coordinates"])
crop_samples["x"] = crop_samples["coords"].apply(lambda x: x[0])
crop_samples["y"] = crop_samples["coords"].apply(lambda x: x[1])
crop_samples = crop_samples[["stable_crop", "x", "y"]]

(4110, 12)
(4108, 12)
(1471, 4)
(1898, 11)
(1875, 11)


In [14]:
samples = pd.concat([crop_samples, noncrop_balanced_samples.drop(columns=["lc2022"])])
samples = geemap.df_to_ee(samples, longitude="x", latitude="y")

In [16]:
geemap.ee_export_vector_to_asset(samples, description='crop_nonCrop_training_samples_glad_re', assetId='projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad')

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad
Exporting crop_nonCrop_training_samples_glad_re... Please check the Task Manager from the JavaScript Code Editor.


## Generate training data with predictors

In [1]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config
from helpers import landsat_composites 
import pandas as pd
import ast

In [ ]:
import plotext

plotext.bar()

In [2]:
start_date = "1999-01-01"
end_date = "2023-12-31"
crs = "EPSG:4326"
scale = 30
roi = ee.FeatureCollection("projects/ee-joshisur231/assets/pa_effectiveness/nepal_boundary")

In [3]:
def prepare_terrain_images(dem, roi, scale):
    terrain_image = ee.Terrain.products(dem)
    slope_norm = terrain_image.select('slope')
    aspect_rad = terrain_image.select('aspect').multiply(3.14159).divide(180)
    
    northness = aspect_rad.cos().rename('northness')
    
    eastness = aspect_rad.sin().rename('eastness')

    return dem.rename("elev").addBands(slope_norm).addBands(northness).addBands(eastness)
terrain_images = prepare_terrain_images(ee.Image('USGS/SRTMGL1_003'), roi, scale)

In [4]:
samples = ee.FeatureCollection('projects/ee-joshisur231/assets/agriculture_abandonment_nepal/crop_nonCrop_training_samples_glad')
processor = landsat_composites.C2SRExpressions()
unified_collection = processor._expression()\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["NDVI"]).rename("ndvi")))\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["EVI"]).rename("evi")))\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["NDMI"]).rename("ndmi")))\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["MSAVI"]).rename("msavi")))

In [5]:
combined_reducer = ee.Reducer.mean() \
    .combine(ee.Reducer.median(), sharedInputs=True) \
    .combine(ee.Reducer.stdDev(), sharedInputs=True) \
    .combine(ee.Reducer.percentile([25, 75]), sharedInputs=True)

In [6]:
def get_3yr_predictors(target_year):
    target_year = ee.Number(target_year)
    start_date = ee.Date.fromYMD(target_year.subtract(1), 1, 1)
    end_date = ee.Date.fromYMD(target_year.add(1), 12, 31)

    window_col = unified_collection.filterDate(start_date, end_date).filterBounds(roi)

    predictors = window_col.reduce(combined_reducer)

    glcm_image = predictors.select("ndvi_mean").max(0)\
        .reproject(crs="EPSG:32645", scale=30)\
        .multiply(32).int32()\
        .glcmTexture(size=1, average=True)
    return  predictors.addBands(glcm_image)

    return predictors.set('year', target_year)\
                     .set('system:time_start', ee.Date.fromYMD(target_year, 1, 1).millis())

In [9]:
for year in range(2000, 2023):
    l_image = get_3yr_predictors(year).addBands(terrain_images)
    asset_id = f"projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_{str(int(year))}"
    
    training_data = l_image.reduceRegions(
        collection=samples,
        scale=scale,
        reducer=ee.Reducer.first(),
        tileScale = 4
    )
    geemap.ee_export_vector_to_asset(
        training_data, 
        description= f'export_train_{str(int(year))}', 
        assetId=asset_id
    )
    geemap.ee_export_vector_to_drive(
        training_data, 
        description= f'trainSamp_{str(int(year))}', 
        fileFormat = "csv",
        folder="aal"
    )
    # geemap.ee_export_image_to_asset(
    #     l_image, 
    #     description= f'export_train_image_{str(int(year))}', 
    #     assetId=asset_id,
    #     scale=scale,
    #     crs=crs,
    #     maxPixels=1e13,
    # )
print("Done!")

projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2000
Exporting export_train_2000... Please check the Task Manager from the JavaScript Code Editor.
Exporting trainSamp_2000... Please check the Task Manager from the JavaScript Code Editor.
projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2001
Exporting export_train_2001... Please check the Task Manager from the JavaScript Code Editor.
Exporting trainSamp_2001... Please check the Task Manager from the JavaScript Code Editor.
projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2002
Exporting export_train_2002... Please check the Task Manager from the JavaScript Code Editor.
Exporting trainSamp_2002... Please check the Task Manager from the JavaScript Code Editor.
projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples_glad/trainSamp_2003
Exporting export_train_2003... Please check the Task 

### Hyperparameter tuning

In [1]:
import pandas as pd
# import numpy as np
# from sklearn.model_selection import KFold, cross_val_score
# # from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import train_test_split, RandomizedSearchCV
# from sklearn.metrics import classification_report
# from sklearn.ensemble import RandomForestClassifier
# import ee
# ee.Initialize(project="ee-joshisur231")
# import geemap
# Map = geemap.Map()

In [2]:
samples = pd.read_csv(r"E:\work\Agricultural Land Abandonment\code\outputs\random_forest\training_data\trainSamp_2021.csv")

Index(['system:index', 'blue_mean', 'blue_median', 'blue_p25', 'blue_p75',
       'blue_stdDev', 'eastness', 'elev', 'evi_mean', 'evi_median', 'evi_p25',
       'evi_p75', 'evi_stdDev', 'green_mean', 'green_median', 'green_p25',
       'green_p75', 'green_stdDev', 'msavi_mean', 'msavi_median', 'msavi_p25',
       'msavi_p75', 'msavi_stdDev', 'ndmi_mean', 'ndmi_median', 'ndmi_p25',
       'ndmi_p75', 'ndmi_stdDev', 'ndvi_mean', 'ndvi_mean_asm',
       'ndvi_mean_contrast', 'ndvi_mean_corr', 'ndvi_mean_dent',
       'ndvi_mean_diss', 'ndvi_mean_dvar', 'ndvi_mean_ent', 'ndvi_mean_idm',
       'ndvi_mean_imcorr1', 'ndvi_mean_imcorr2', 'ndvi_mean_inertia',
       'ndvi_mean_maxcorr', 'ndvi_mean_prom', 'ndvi_mean_savg',
       'ndvi_mean_sent', 'ndvi_mean_shade', 'ndvi_mean_svar', 'ndvi_mean_var',
       'ndvi_median', 'ndvi_p25', 'ndvi_p75', 'ndvi_stdDev', 'nir_mean',
       'nir_median', 'nir_p25', 'nir_p75', 'nir_stdDev', 'northness',
       'red_mean', 'red_median', 'red_p25', 'red_p75',

In [ ]:
predictors = ['blue_mean', 'blue_median', 'blue_p25', 'blue_p75', 'blue_stdDev',
       'green_mean', 'green_median', 'green_p25', 'green_p75', 'green_stdDev',
       'ndvi_mean', 'ndvi_median', 'ndvi_p25', 'ndvi_p75', 'ndvi_stdDev',
       'nir_mean', 'nir_median', 'nir_p25', 'nir_p75', 'nir_stdDev',
       'red_mean', 'red_median', 'red_p25', 'red_p75', 'red_stdDev',
       'swir1_mean', 'swir1_median', 'swir1_p25', 'swir1_p75',
       'swir1_stdDev', 'swir2_mean', 'swir2_median', 'swir2_p25', 'swir2_p75',
       'swir2_stdDev']
target = "stable_crop"
X = samples[predictors]
y = samples[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=45)

param_grid = {
    'n_estimators': [50, 100, 200, 300, 400, 500, 800], 
    'max_features': ['sqrt', 'log2'],
    # 'max_depth': [None, 10, 20, 30, 40, 50], 
    # 'min_samples_split': [2, 5, 10, 15], 
    'min_samples_leaf': [1, 2, 4, 6] 
}

rf_model = RandomForestClassifier(random_state=45)

kfold = KFold(n_splits=10, shuffle=True, random_state=45)

rf_search = RandomizedSearchCV(
    estimator=rf_model, 
    param_distributions=param_grid, 
    n_iter=50, 
    cv=kfold, 
    verbose=4, 
    random_state=45, 
    n_jobs=-1,
    scoring="f1"
)
rf_search.fit(X_train, y_train)

y_pred = rf_search.predict(X_test)

print(rf_search.best_params_)
print(rf_search.best_estimator_)
print(f"\nClassification Report")
print(classification_report(y_test, y_pred))

importance_df = pd.DataFrame({'Feature': predictors, 'Importance': rf_search.best_estimator_.feature_importances_})
importance_df["relative importance"] = importance_df["Importance"] * 100 / importance_df["Importance"].sum() 
print(importance_df.sort_values(by='Importance', ascending=False))
results_df = pd.DataFrame(rf_search.cv_results_)

columns_to_view = [
    'param_n_estimators', 
    # 'param_max_depth', 
    'mean_test_score',      
    'std_test_score',       
    'rank_test_score'       
]

best_models = results_df[columns_to_view].sort_values(by='rank_test_score').head(5)

print("\nBest Models")
print(best_models)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'n_estimators': 50, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
RandomForestClassifier(n_estimators=50, random_state=45)

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1114
           1       0.99      0.97      0.98       550

    accuracy                           0.99      1664
   macro avg       0.99      0.98      0.99      1664
weighted avg       0.99      0.99      0.99      1664

         Feature  Importance  relative importance
8      green_p75    0.159902            15.990227
5     green_mean    0.143722            14.372186
6   green_median    0.099703             9.970320
33     swir2_p75    0.086517             8.651655
20      red_mean    0.047139             4.713865
28     swir1_p75    0.043880             4.387974
11   ndvi_median    0.040185             4.018486
10     ndvi_mean    0.039972             3.997236
34  swir2_stdDe

In [14]:
pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T

,precision,recall,f1-score,support
0,0.985778,0.995512,0.990621,1114.00000
1,0.990724,0.970909,0.980716,550.00000
accuracy,0.987380,0.987380,0.987380,0.98738
macro avg,0.988251,0.983210,0.985669,1664.00000
weighted avg,0.987413,0.987380,0.987347,1664.00000


In [13]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1114
           1       0.99      0.97      0.98       550

    accuracy                           0.99      1664
   macro avg       0.99      0.98      0.99      1664
weighted avg       0.99      0.99      0.99      1664



## Making prediction

In [ ]:
import ee
ee.Initialize(project="ee-joshisur231")
import geemap
import helpers.config as config
from helpers import landsat_composites 
import pandas as pd
import ast
import math

In [ ]:
predictors = ['blue_mean', 'blue_median', 'blue_p25', 'blue_p75', 'blue_stdDev',
       'green_mean', 'green_median', 'green_p25', 'green_p75', 'green_stdDev',
       'ndvi_mean', 'ndvi_median', 'ndvi_p25', 'ndvi_p75', 'ndvi_stdDev',
       'nir_mean', 'nir_median', 'nir_p25', 'nir_p75', 'nir_stdDev',
       'red_mean', 'red_median', 'red_p25', 'red_p75', 'red_stdDev',
       'swir1_mean', 'swir1_median', 'swir1_p25', 'swir1_p75',
       'swir1_stdDev', 'swir2_mean', 'swir2_median', 'swir2_p25', 'swir2_p75',
       'swir2_stdDev']
samples_train = ee.FeatureCollection("projects/ee-joshisur231/assets/agriculture_abandonment_nepal/training_samples/train_2022").filter(ee.Filter.notNull(predictors))
processor = landsat_composites.C2SRExpressions()
unified_collection = processor._expression()\
    .map(lambda image: image.addBands(image.expression(processor.ALGORITHMS["NDVI"]).rename("ndvi")))

In [ ]:
combined_reducer = ee.Reducer.mean() \
    .combine(ee.Reducer.median(), sharedInputs=True) \
    .combine(ee.Reducer.stdDev(), sharedInputs=True) \
    .combine(ee.Reducer.percentile([25, 75]), sharedInputs=True)

In [ ]:
def get_3yr_predictors(target_year):
    target_year = ee.Number(target_year)
    start_date = ee.Date.fromYMD(target_year.subtract(1), 1, 1)
    end_date = ee.Date.fromYMD(target_year.add(1), 12, 31)

    window_col = unified_collection.filterDate(start_date, end_date).filterBounds(config.ROI)

    predictors = window_col.reduce(combined_reducer)

    return predictors.set('year', target_year) \
                     .set('system:time_start', ee.Date.fromYMD(target_year, 1, 1).millis())

In [ ]:
l_2022 = get_3yr_predictors(2022).clip(config.ROI)

In [ ]:
# {'n_estimators': 100, 'min_samples_leaf': 2, 'max_features': 'log2'}
rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees= 100, #n_estimators
    minLeafPopulation = 2,#min_samples_leaf
    variablesPerSplit = round(math.log2(len(predictors))), #max_features
)\
    .setOutputMode('MULTIPROBABILITY')\
    .train(features = samples_train, classProperty="stable_crop", inputProperties=predictors)

classified_2022 = l_2022.classify(rf_classifier)
probabilities = classified_2022.arrayFlatten([["noncrop_prob", "crop_prob"]])

In [ ]:
Map.addLayer(probabilities.select("crop_prob"), {"min":0, "max": 1, "palette":["red", "yellow", "green"]}, "prob")
Map

In [ ]:
def hyperparameter_tuning(samples, predictor_names, actual_class_col, split_frac, n_trees):
    samples = samples.randomColumn()
    train_samples = samples.filter(ee.Filter.lt("random", split_frac))
    val_samples = samples.filter(ee.Filter.gte("random", split_frac))

    def compute_acc(n_tree):
        n_tree = ee.Number(n_tree)
        classifier = ee.Classifier.smileRandomForest(n_tree)\
            .train(features=train_samples, classProperty=actual_class_col, inputProperties=predictor_names)

        error_mat = val_samples\
            .classify(classifier)\
            .errorMatrix(actual_class_col, "classification")
        
        acc = error_mat.accuracy()
        p_acc = error_mat.producersAccuracy()
        u_acc = error_mat.consumersAccuracy()
        f_score = error_mat.fscore()
        kappa = error_mat.kappa()

        acc_dict = {
            "error_matrix": error_mat.array(),
            "accuracy": acc,
            "kappa": kappa,
            "f": f_score,
            "producer": p_acc,
            "user": u_acc
         }

        return acc_dict
    
    accuracies = n_trees.map(compute_acc)
    return accuracies


In [ ]:
hyperparameter_tuning(samples, predictors, "stable_crop", 0.7, ee.List([50, 150, 250, 350])).getInfo()

In [1]:
{-1014453005: 25
-1019554964: 14.819607843137254
-1022696823: 8
-1078166501: 25
-1079969722: 23
-1081318159: 30
-1122116466: 9
-1123415199: 4.098039215686274
-1132402806: 9
-113451068: 24.411764705882355
-114429426: 13.611764705882353
-1155835179: 38
-1182844551: 7
-1197018590: 1.3764705882352941
-1207703862: 4.3803921568627455
-1229338026: 3
-1240928115: 20
-1283156731: 23.454901960784312
-1299835362: 9.894117647058824
-132120915: 0.03529411764705882
-1324974404: 5
-1367134926: 1.7098039215686274
-1397371651: 7
-1433440992: 1.8196078431372549
-1445488938: 5.556862745098039
-1450676523: 2
-1473746044: 16
-1499697462: 10
-1540377753: 19
-154647614: 22
-1567369310: 1.6666666666666665
-1601585073: 13
-1626197325: 23.211764705882356
-1629389684: 0.16862745098039217
-1651118055: 16
-1671504397: 25.007843137254902
-167350268: 9
-168099920: 17
-1685794051: 33.76862745098039
-1688492082: 17.615686274509805
-1688867066: 18
-1694562478: 24
-1717335106: 15.898039215686275
-1718001757: 26.65490196078431
-1742460038: 5
-1751737637: 1.1176470588235294
-1767026652: 3.588235294117647
-1769186422: 2.2941176470588234
-1769922136: 14
-1779770995: 29
-1810045914: 9
-1842702663: 12
-1848936714: 18
-1873688977: 6.047058823529411
-1874270601: 6
-1876842150: 9.898039215686275
-1890279506: 10.239215686274509
-1895686070: 9
-1898264199: 0.08235294117647059
-1905563146: 15
-190585919: 22
-1913968465: 16
-1988707108: 0.10980392156862745
-2006121602: 17
-20078654: 4.890196078431373
-2009963012: 18
-2034556943: 13.180392156862744
-2042764022: 10
-2064940064: 14
-2069743131: 14
-2077158961: 4
-2120287679: 5
-213309292: 1
-2135273892: 9.105882352941176
-224253120: 3.7490196078431373
-243346331: 0.03529411764705882
-247435394: 21
-248007519: 11
-254884537: 4.949019607843137
-324415868: 6
-325005691: 6
-325090714: 20
-328928222: 11
-340627820: 17
-3426497: 1.9529411764705882
-346507068: 24
-375019317: 29
-377775693: 11
-396805265: 1.3490196078431371
-411018853: 16.901960784313726
-431650596: 18
-445614288: 12
-447697812: 18.91764705882353
-451468309: 6.0470588235294125
-479553409: 0.011764705882352941
-49424888: 34.64705882352941
-497391018: 18
-49771090: 6
-497980187: 7
-539070137: 13.192156862745097
-550420654: 12
-559441379: 21
-621898230: 5.835294117647059
-73705061: 15
-740602288: 31
-75021275: 30
-752221913: 4
-757496917: 10
-763773503: 20
-772670083: 7
-774144880: 13
-819670346: 6
-869548694: 33
-871533316: 2
-91960837: 19
-958826021: 24
-967473373: 5.717647058823529
-97052618: 12
-982674988: 12.623529411764707
-988636759: 21
1006461177: 22.854901960784314
1031532718: 18
1035772586: 25
1041852318: 0.23921568627450981
1066770832: 2.545098039215686
1073217671: 8
1092321801: 6.4941176470588236
1121042765: 25
1127900281: 14
117780412: 5.509803921568627
1181326227: 27
1193584462: 15
1203292538: 14
1208382468: 9
1239463997: 11
128031805: 1.423529411764706
1282425892: 9
1297135225: 1
1304762741: 1.83921568627451
1316523956: 16
1336721494: 2.145098039215686
1356938960: 5.83921568627451
1367795304: 12.301960784313726
138517721: 14
1401523701: 16
1432428026: 18
1457862781: 24
1458271376: 15
146197267: 16
1466239604: 12.580392156862747
1481266496: 0.6352941176470588
1486777705: 19.164705882352944
1494649663: 0.0392156862745098
1524270343: 2
153988723: 11
1562236107: 28
1564842588: 8
15836015: 10.67450980392157
1589047048: 16
1598196533: 29
1599550324: 19
1645574365: 8
1646027602: 16.254901960784313
1653607682: 13
1681348390: 8.615686274509804
1688252940: 9
1727702821: 15.070588235294117
1753052367: 9
176033869: 10
1773944166: 11
1785518011: 13
1790236404: 33
1800510501: 13
1801624146: 12
1821532555: 13
182502101: 20
1828416272: 18
1830646180: 21
186501106: 20
1870761954: 21
1882148449: 9.96078431372549
1886805595: 14
1895469176: 33.07058823529412
1905059786: 23
1912090072: 14.780392156862746
1914200394: 13.196078431372548
1932757574: 20
1949449035: 16.83137254901961
1975695602: 11
2015404388: 9
2055973334: 0.4666666666666667
2058273930: 8
207626888: 5.337254901960785
2117534783: 12
2125619585: 10
216355726: 0.00392156862745098
222143369: 5
233942119: 19.231372549019607
257726930: 27
258630878: 9.274509803921568
260571254: 1.4352941176470588
265013948: 15
26838253: 30
278743084: 8
278832627: 18
325518505: 22
334629027: 17
353086427: 16
360358433: 8
376404379: 2.635294117647059
466183342: 24
4807773: 2.317647058823529
494445947: 6
499952777: 12.015686274509804
526975085: 0.780392156862745
535785515: 8.427450980392157
540379394: 24
560846078: 11
56167839: 18
592363356: 14
595742956: 21
599098561: 19
604404155: 19
612362219: 6
616920838: 13
616945987: 17
664788051: 4.223529411764706
703088499: 10
709779755: 11
711445493: 1.619607843137255
714800828: 11
714971585: 0.788235294117647
744830374: 1.0235294117647058
760348542: 0.9490196078431372
763600402: 20
782526445: 9
783818123: 1.5568627450980392
801932837: 6
815239593: 7
83634643: 22
893847561: 22
930548061: 17
947883448: 19
973718811: 8.113725490196078
987360020: 17
-1030219473: 31
-1080142676: 5.301960784313725
-1083726475: 65
-1084791529: 15.058823529411764
-1100649377: 79.87058823529414
-117776021: 124
-1227085324: 115
-1233113689: 132.12549019607843
-1246716768: 134.46274509803922
-1262502304: 198
-1342840497: 99
-1391829420: 179.09803921568627
-1424859900: 65
-1456363320: 93
-1471509183: 38
-1476034575: 187.84313725490196
-149632207: 259
-1498655684: 325.3921568627451
-1517319020: 3.831372549019608
-1537094710: 4.627450980392156
-1634486991: 135.46666666666667
-1702637754: 17.082352941176467
-1717702126: 117.44313725490196
-178190947: 3.0666666666666664
-1784554175: 107.01176470588236
-1806841624: 2.0705882352941174
-1857545168: 121
-1899819896: 1.1098039215686275
-1926002606: 34.42352941176471
-1926472814: 1.1019607843137256
-1928245714: 62.02352941176471
-1942714412: 13.525490196078431
-1970044061: 114.15686274509804
-2051839047: 74.45098039215686
-206011908: 72.17647058823529
-21263424: 36
-236855556: 122.98823529411764
-261245619: 49
-282596926: 243
-299493549: 35.984313725490196
-32575125: 13.654901960784313
-328556686: 239
-329859455: 78
-339950585: 27.30980392156863
-363497149: 143.09019607843138
-38256602: 59.38823529411764
-411127771: 180.45490196078433
-497187398: 249
-533777462: 129.85098039215686
-57027881: 93.89019607843137
-584201887: 14
-596791129: 3.3568627450980397
-614953314: 54
-645371480: 124.05098039215686
-700237837: 48
-795225591: 60
-825137831: 5.717647058823529
-834094198: 52
-848068204: 0.34901960784313724
-882651508: 32.43529411764706
-883360448: 32.54117647058824
-941529383: 146
-960266343: 85.81176470588235
-967340489: 110
-993502016: 57.16470588235294
1043324085: 202.28235294117647
1091661646: 10.882352941176471
1153698873: 216.643137254902
1334117137: 39
136927303: 140.67450980392155
1398109481: 61.32156862745098
1438046403: 219.25882352941176
1441198320: 5.698039215686275
1450726899: 135.74901960784314
1455811989: 0.5647058823529412
1477662168: 2.364705882352941
1505851738: 21.274509803921568
1540667269: 150.87450980392157
1560816146: 131
1600905072: 2.231372549019608
1601353226: 95.30196078431372
1645668760: 19.60392156862745
1654442427: 137.39999999999998
1668421428: 26.23921568627451
1693269257: 54.68627450980392
1745906437: 118
1811026633: 25.67843137254902
1834907100: 0.20784313725490194
1839950724: 24.380392156862744
1882491226: 56.811764705882354
1893349962: 86
190422413: 163.91372549019607
1913847592: 241
2075022664: 124
2095595044: 155
2130526079: 53.599999999999994
2134174836: 345.1725490196078
215394163: 10.305882352941177
291757022: 185.14509803921567
318934190: 60.43529411764706
434042145: 61.52941176470589
503018302: 109
520163633: 6.352941176470588
565912884: 30.419607843137253
569415904: 14.941176470588234
599671066: 32.80392156862745
604488947: 48.756862745098054
621450832: 204
658832989: 1.6392156862745098
675295493: 206.38039215686274
684927782: 218
695280424: 2.015686274509804
721227544: 227
721823962: 30.584313725490198
769172515: 91
780477991: 63
853092435: 41.91372549019607
877787723: 292.6
900673536: 200
957203596: 0.08627450980392157
961572211: 106}

SyntaxError: invalid syntax (3299352139.py, line 2)